# Network Flow V2 — Edge-IIoTset Step 02: Feature and Data Quality Analysis

**Input:** `data/raw/edge_iiotset/ML-EdgeIIoT-dataset.csv` (your 157,800-row ML selection).  
**Purpose:** verify labels, duplicates, suspicious zero-filled columns, mixed data types, and feature meanings **before any train/validation/test split**.

This notebook is **read-only with respect to the original CSV**. It saves only aggregated metadata (no raw traffic, IP addresses, or labels per record) to `reports/` and a feature dictionary draft to `docs/`.

**Why now?** The Step 01 audit found 133,499 attacks, 24,301 normal records, and several fields whose representations need verification. "0 missing values" does **not** mean all feature values are meaningful.

> Run all cells in order using your V2 `.venv` Jupyter kernel. Do **not** train a model or fit encoders/scalers here.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import pandas as pd
import numpy as np
from IPython.display import display

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
DATA = ROOT / "data" / "raw" / "edge_iiotset" / "ML-EdgeIIoT-dataset.csv"
REPORT_DIR = ROOT / "reports" / "audits"
DOCS = ROOT / "docs"
FIGURES = ROOT / "reports" / "figures"

assert DATA.is_file(), f"CSV not found: {DATA}"
print("Reading:", DATA.name)
df = pd.read_csv(DATA, low_memory=False)
print(f"Rows: {len(df):,}; columns: {len(df.columns)}; RAM used by DataFrame: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print("Labels present:", [c for c in ("Attack_label", "Attack_type") if c in df.columns])
assert {"Attack_label", "Attack_type"}.issubset(df.columns), "Stop: expected label columns missing"
assert df.columns.is_unique, "Stop: duplicated column names"
FEATURES = [c for c in df.columns if c not in ("Attack_label", "Attack_type")]
print("Candidate non-label columns:", len(FEATURES))


## 1. Verify binary and attack-type label consistency

**Ground truth:** the correct attack/normal and attack-type values provided in the dataset. Both are for supervised training targets and evaluation, **never model features**.

A mismatch means an observation's binary label disagrees with whether its `Attack_type` is `Normal`. Such rows need review before modelling.

In [ ]:
binary = pd.to_numeric(df["Attack_label"], errors="coerce")
type_clean = df["Attack_type"].astype("string").str.strip()
expected_attack = type_clean.str.casefold().ne("normal")
valid = binary.isin([0, 1]) & type_clean.notna() & type_clean.ne("")
label_mismatch = valid & binary.ne(expected_attack.fillna(False).astype("int64"))

label_summary = pd.DataFrame({
    "Attack_label": [0, 1],
    "Meaning": ["Normal", "Attack"],
    "Count": [int((binary == 0).sum()), int((binary == 1).sum())],
})
label_summary["Percent"] = (100 * label_summary["Count"] / len(df)).round(3)
display(label_summary)
print("Invalid/missing binary or attack-type labels:", int((~valid).sum()))
print("Binary versus attack-type mismatches:", int(label_mismatch.sum()))
print("Number of distinct attack types (excluding Normal):", int(type_clean[type_clean.str.casefold() != "normal"].nunique()))

class_summary = type_clean.value_counts(dropna=False).rename_axis("attack_type").reset_index(name="count")
class_summary["percent"] = (class_summary["count"] / len(df) * 100).round(3)
display(class_summary)


## 2. Raw duplicates and identical feature combinations

- **Exact duplicate**: all 63 columns are identical across observations.
- **Same feature combination**: the 61 non-label fields are identical, whether or not labels differ.
- **Conflicting labels**: identical candidate inputs appear with both Attack and Normal labels; this can make classification impossible for those features.

The hash-based check is a fast **screening test**, not formal proof of exact equality. Do **not** automatically remove duplicates: real repeated traffic may be legitimate.

In [ ]:
exact_duplicate_rows = int(df.duplicated(keep=False).sum())
exact_duplicate_excess = int(df.duplicated(keep="first").sum())
feature_duplicate_rows = int(df.duplicated(subset=FEATURES, keep=False).sum())

# Hash the candidate feature values for a quick conflict screening.
feature_fingerprint = pd.util.hash_pandas_object(df[FEATURES], index=False)
conflict_probe = pd.DataFrame({"fingerprint": feature_fingerprint.to_numpy(), "binary": binary.to_numpy()})
conflicting_fingerprints = conflict_probe.groupby("fingerprint", sort=False)["binary"].nunique(dropna=True)
conflict_ids = conflicting_fingerprints[conflicting_fingerprints > 1].index
conflicting_feature_rows = int(conflict_probe["fingerprint"].isin(conflict_ids).sum())

print(f"Rows belonging to exact-duplicate groups: {exact_duplicate_rows:,}")
print(f"Excess exact copies beyond first occurrence: {exact_duplicate_excess:,}")
print(f"Rows sharing all non-label feature values: {feature_duplicate_rows:,}")
print(f"Rows in feature-fingerprint groups with conflicting binary labels: {conflicting_feature_rows:,}")
print("No rows were dropped or modified.")


## 3. Detect sparse / zero-heavy columns and missing-value markers

Some fields are protocol-specific. For example, MQTT-only fields may be zero for HTTP traffic. **Zero is not necessarily a missing value**. This cell reports suspicious sparsity without changing it.

In [ ]:
placeholder_tokens = {"", "?", "na", "n/a", "nan", "none", "null", "undefined", "inf", "-inf", "-"}
feature_rows=[]
for name in FEATURES:
    s=df[name]
    numeric=pd.api.types.is_numeric_dtype(s)
    zero_pct = float((s == 0).mean()*100) if numeric else None
    marker_count=0
    if not numeric:
        normalized=s.astype("string").str.strip().str.casefold()
        marker_count=int(normalized.isin(placeholder_tokens).sum())
    feature_rows.append({
        "feature":name,
        "raw_dtype":str(s.dtype),
        "null_count":int(s.isna().sum()),
        "distinct_values":int(s.nunique(dropna=True)),
        "zero_pct":round(zero_pct,2) if zero_pct is not None else None,
        "placeholder_token_count":marker_count,
    })
quality=pd.DataFrame(feature_rows)
print("Columns >=95% numeric zeros (inspect; do not automatically drop):")
display(quality[quality["zero_pct"].ge(95)].sort_values("zero_pct", ascending=False))
print("String columns or columns containing missing-value tokens:")
display(quality[(quality["raw_dtype"] == "object") | (quality["raw_dtype"].isin(["string", "str"])) | (quality["placeholder_token_count"] > 0)])
print("Potentially constant columns:")
display(quality[quality["distinct_values"] <= 1][["feature", "raw_dtype", "distinct_values"]])


## 4. Investigate suspicious field representations without printing raw addresses

Do not assume `frame.time` contains valid chronological timestamps just because it is numeric. IP addresses and ports may have been transformed by the dataset authors. Check source documentation before interpreting their units or including them in a model.

The table intentionally avoids printing IP addresses, host IDs, or payload contents.

In [ ]:
focus = ["frame.time", "ip.src_host", "ip.dst_host", "tcp.srcport", "tcp.dstport", "http.file_data", "mqtt.msg"]
rows=[]
for name in focus:
    if name not in df.columns:
        continue
    s=df[name]
    numeric_view=pd.to_numeric(s, errors="coerce")
    rows.append({
        "feature":name,
        "raw_dtype":str(s.dtype),
        "distinct_values":int(s.nunique(dropna=True)),
        "numeric_parse_pct":round(100*numeric_view.notna().mean(),2),
        "numeric_zero_pct":round(100*(numeric_view==0).mean(),2),
        "has_fractional_numeric_values": bool((numeric_view.dropna().mod(1) != 0).any()),
    })
focus_summary=pd.DataFrame(rows)
display(focus_summary)
print("Interpretations and units remain UNVERIFIED until matched to release documentation.")


## 5. Make a draft feature dictionary

A **feature dictionary** explains what each input column represents, how it is stored, and whether its meaning matches a feature in another dataset. This is the basis for your later Edge-IIoTset / X-IIoTID / ToN-IoT common-feature mapping.

The groups below come **only from the column names**. They are organizational hints, not proof of the feature definitions.

In [ ]:
def family_of(name):
    return name.split(".",1)[0] if "." in name else "other"

feature_dictionary = quality.copy()
feature_dictionary.insert(1, "candidate_family", [family_of(x) for x in feature_dictionary["feature"]])
feature_dictionary["verified_unit"] = "UNVERIFIED"
feature_dictionary["live_gateway_extractable"] = "TO_CHECK"
feature_dictionary["label_leakage_checked"] = "NO"
feature_dictionary["cross_dataset_equivalent"] = "NOT_MAPPED"
display(feature_dictionary.groupby("candidate_family")["feature"].count().rename("candidate_features").to_frame())
display(feature_dictionary.head(12))


## 6. Optional picture of class distribution

This is descriptive only: no model has been trained. The CSV has 15 classes (14 attacks plus Normal), but the classes are not equally represented.

In [ ]:
import matplotlib.pyplot as plt
FIGURES.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(class_summary["attack_type"].astype(str)[::-1], class_summary["count"][::-1])
ax.set_xlabel("Records in your selected ML CSV")
ax.set_title("Edge-IIoTset ML subset — per-class records")
fig.tight_layout()
fig_path=FIGURES / "edge_iiotset_class_distribution_v1.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Figure:",fig_path)


## 7. Save aggregated findings and a dictionary draft

Safe to commit after reviewing the file content: only counts, field names, and aggregate statistics are saved. The raw dataset is untouched.

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)
report={
    "dataset":DATA.name,
    "rows":int(len(df)),
    "columns":int(len(df.columns)),
    "feature_candidates":int(len(FEATURES)),
    "binary_counts":{str(k):int(v) for k,v in binary.value_counts(dropna=False).items()},
    "attack_type_counts":{str(k):int(v) for k,v in type_clean.value_counts(dropna=False).items()},
    "label_invalid_rows":int((~valid).sum()),
    "label_mismatch_rows":int(label_mismatch.sum()),
    "exact_duplicate_group_rows":exact_duplicate_rows,
    "exact_duplicate_excess_rows":exact_duplicate_excess,
    "feature_duplicate_group_rows":feature_duplicate_rows,
    "potential_conflicting_binary_label_rows_hash_screen":conflicting_feature_rows,
    "columns_at_least_95pct_zero":quality.loc[quality["zero_pct"].ge(95),"feature"].tolist(),
    "constant_columns":quality.loc[quality["distinct_values"]<=1,"feature"].tolist(),
    "mixed_dtype_review":rows,
    "note":"Read-only audit; duplicate and hash collisions require provenance-aware review. No split, normalization, or training was performed."
}
json_path=REPORT_DIR / "edge_iiotset_feature_quality_v1.json"
json_path.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding="utf-8")

def escape_md(x):
    return str(x).replace("|", "\\|").replace("\n"," ")
header=["Feature", "Family (candidate)", "Raw dtype", "Distinct", "Zero %", "Verified meaning/unit", "Gateway-compatible?"]
lines=["# Edge-IIoTset — Feature dictionary DRAFT", "", "> Column groups are inferred from names; meanings, units and live-capture feasibility remain unverified.", "", " | ".join(header), " | ".join(["---"]*len(header))]
for r in feature_dictionary.to_dict(orient="records"):
    vals=[r["feature"],r["candidate_family"],r["raw_dtype"],r["distinct_values"], r["zero_pct"] if pd.notna(r["zero_pct"]) else "N/A", "UNVERIFIED", "TO_CHECK"]
    lines.append(" | ".join(escape_md(v) for v in vals))
md_path=DOCS / "EDGE_IIOTSET_FEATURE_DICTIONARY_DRAFT.md"
md_path.write_text("\n".join(lines)+"\n",encoding="utf-8")
print("Quality report:",json_path)
print("Feature dictionary:",md_path)


## Stop here — do not split yet

Share these Step 02 outputs:
1. **Invalid/mismatched label counts**
2. **Exact duplicate rows, feature-only duplicate rows, and possible conflicting-label rows**
3. **Zero-heavy/constant columns**
4. **Suspicious-field representation table** (`frame.time`, IP, TCP fields)

**Then** we can choose a defensible 70% train / 15% validation / 15% test strategy. If capture/session identifiers are absent in this selected CSV, we must describe any fallback grouping method and its limitations. We cannot claim the result proves generalization to an unseen APT campaign.

We will postpone shared-feature alignment and ensemble training until this audit is reviewed.